# Regional aggregation-threshold sensitivity (multi-cell, in-domain)

Re-runs the multi-cell **in-domain** phase with the same settings as the published run
(3 seeds, 60 epochs, 200 RF trees, RF `max_features="sqrt"`, levels D0–D3) and scores the regional
alert at **20%, 30%, 40% and 50%** of cells. It also saves the daily SPEI/SSI of observations and of every
forecast, so any other threshold can be scored later **without retraining**.

**Estimated time:** about 135 min in total (the published in-domain run took 135.5 min), split into
four horizons of roughly 30–40 min. Each horizon is saved as it finishes, so an interrupted session
loses at most one horizon.

**Before running:** Settings → Accelerator → **GPU**; Environment → **Always use latest environment**.
Attach one dataset created from `Forecast_skill_is_not_warning_skill.zip`; it already contains the code,
the cell data and the published `results_mc` folder used to validate the 30% results.

In [ ]:
# 0 | Check the attached dataset (the uploaded Forecast_skill_is_not_warning_skill.zip)
import glob
need = {'tcn_drought_multicell.py': '/kaggle/input/**/tcn_drought_multicell.py',
        'tcn_drought_pipeline_v2.py': '/kaggle/input/**/tcn_drought_pipeline_v2.py',
        'cells.csv': '/kaggle/input/**/cells.csv',
        'cell_*.csv (10 files)': '/kaggle/input/**/cell_*.csv',
        'results_mc/dss_regional.csv (optional, for validation)': '/kaggle/input/**/results_mc/dss_regional.csv'}
for label, pat in need.items():
    found = glob.glob(pat, recursive=True)
    print(f"{'OK ' if found else '-- '} {label:55s} {len(found)} found  {found[0] if found else ''}")


In [ ]:
# 1 | Scripts, working directory and GPU check
import os, sys, glob, shutil
hits = glob.glob('/kaggle/input/**/tcn_drought_multicell.py', recursive=True)
assert hits, "Attach the dataset made from Forecast_skill_is_not_warning_skill.zip via 'Add Input'."
assert os.path.exists(os.path.join(os.path.dirname(hits[0]), 'tcn_drought_pipeline_v2.py')), \
    'tcn_drought_pipeline_v2.py must sit next to tcn_drought_multicell.py'
for f in glob.glob(os.path.join(os.path.dirname(hits[0]), '*.py')): shutil.copy(f, '/kaggle/working/')
os.chdir('/kaggle/working'); sys.path.insert(0, '/kaggle/working')

import tensorflow as tf
gpus = tf.config.list_physical_devices('GPU')
print('TF', tf.__version__, '| GPUs:', gpus)
assert gpus, "STOP: no GPU visible to TensorFlow. Enable the GPU accelerator and use the latest environment."


In [ ]:
# 2 | Data and published results (for validation)
import pandas as pd
cand = [f for f in glob.glob('/kaggle/input/**/cells.csv', recursive=True)
        if len(glob.glob(os.path.join(os.path.dirname(f), 'cell_*.csv'))) >= 10]
assert cand, "Attach the cell-data dataset (cells.csv + cell_*.csv)."
DATA_DIR = os.path.dirname(cand[0]); print('DATA_DIR =', DATA_DIR)
pub = glob.glob('/kaggle/input/**/results_mc/dss_regional.csv', recursive=True)
PUBLISHED = pub[0] if pub else None
print('published dss_regional.csv:', PUBLISHED or 'not attached (validation will be skipped)')
OUT_DIR = '/kaggle/working/results_mc_regional'          # new folder: published files are never overwritten
os.makedirs(OUT_DIR, exist_ok=True)


In [ ]:
%%writefile regional_patch.py
"""
Regional-aggregation-threshold sensitivity for the multi-cell in-domain experiment.

Drop-in replacement for tcn_drought_multicell.run_indomain. Compared with the
released function it
  (1) verifies drought levels D0-D3, as in the published multi-cell run,
  (2) scores the regional alert at several aggregation fractions (FRACS),
  (3) writes, per horizon, the daily SPEI and SSI of the observations and of every
      forecast at every cell, so any other fraction, level or rule can be scored
      later without retraining.
Models, seeds, epochs, scaling, pooling and verification are unchanged.

merge_regional() replaces merge_shards() for this run: merge_shards de-duplicates
regional rows on (horizon, model, level) and would silently drop all but one fraction.
"""
import os, glob
import numpy as np
import pandas as pd
import tcn_drought_multicell as M

P = M.P
LEVELS = (1, 2, 3, 4)                 # D0, D1, D2, D3
FRACS = (0.2, 0.3, 0.4, 0.5)          # share of cells required for a regional alert


def _scores(o, f):
    hits = int((f & o).sum()); fa = int((f & ~o).sum()); miss = int((~f & o).sum())
    cn = int((~f & ~o).sum())
    pod = hits / (hits + miss) if (hits + miss) else np.nan
    far = fa / (hits + fa) if (hits + fa) else np.nan
    csi = hits / (hits + miss + fa) if (hits + miss + fa) else np.nan
    base = (hits + miss) / len(o) if len(o) else np.nan
    return dict(POD=pod, FAR=far, CSI=csi, base_rate=base, hits=hits, false_alarms=fa, misses=miss,
                correct_neg=cn)


def regional_rows(status_obs, status_fc, h, fracs=FRACS, levels=LEVELS):
    """status_obs: DataFrame date x cell (int severity); status_fc: {model: same}."""
    rows = []
    for frac in fracs:
        for level in levels:
            OBS = (status_obs >= level).mean(axis=1) >= frac
            for m, sf in status_fc.items():
                FC = (sf >= level).mean(axis=1) >= frac
                a, b = OBS.align(FC, join="inner")
                rows.append(dict(horizon=h, model=m, logic="or", level=P.LEVEL_NAMES[level], threshold=frac,
                                 **_scores(a.values, b.values)))
    return rows


def run_indomain_regional(cells, cfg):
    ids = list(cells.keys())
    z = M.zscore_static(cells, ids, cfg)
    gi, si = cfg["targets"].index("GWETROOT"), cfg["targets"].index("SPEI_1")
    learned = ["RandomForest"] + (["TCN", "LSTM", "Transformer"] if P.HAS_TF else [])
    rows, dss_pool, reg_rows = [], [], []
    sdir = os.path.join(cfg["out_dir"], "status"); os.makedirs(sdir, exist_ok=True)

    for h in M._horizons(cfg):
        Xtr, Ytr = [], []
        for cid in ids:
            Xw, Yw, _ = M.cell_windows(cells, cid, z, h, cfg, "train"); Xtr.append(Xw); Ytr.append(Yw)
        fitted = M.fit_pool(np.concatenate(Xtr), np.concatenate(Ytr), cfg, learned)

        models_all = ["Persistence", "Climatology"] + learned
        pool_obs = {"spei": [], "ssi": []}
        pool_fc = {m: {"spei": [], "ssi": []} for m in models_all}
        st_obs, st_fc = {}, {m: {} for m in models_all}
        values = []

        for cid in ids:
            Xte, Yte, Ylast = M.cell_windows(cells, cid, z, h, cfg, "test")
            d = cells[cid]["d"]; inv = d["sY"].inverse_transform
            Yte_o, Ylast_o = inv(Yte), inv(Ylast)
            tdates = d["te_index"][cfg["lookback"] + h - 1: cfg["lookback"] + h - 1 + len(Yte)]
            tdoy = np.array([dt.dayofyear for dt in tdates]); clim = cells[cid]["ssi_clim"]

            preds = {"Persistence": {gi: Ylast_o[:, gi], si: Ylast_o[:, si]},
                     "Climatology": {gi: P.climatology_pred(d["tr_index"], d["tr_raw"][:, gi], tdates),
                                     si: P.climatology_pred(d["tr_index"], d["tr_raw"][:, si], tdates)}}
            scaled = M.predict_pool(fitted, Xte)
            for m in learned:
                po = inv(scaled[m]); preds[m] = {gi: po[:, gi], si: po[:, si]}

            for m, pr in preds.items():
                for ti, tname in [(gi, "GWETROOT"), (si, "SPEI_1")]:
                    rows.append(dict(cell_id=cid, horizon=h, target=tname, model=m,
                                     **P.all_metrics(Yte_o[:, ti], pr[ti], Ylast_o[:, ti])))

            idx = pd.DatetimeIndex(tdates)
            oss = P.apply_ssi(Yte_o[:, gi], tdoy, clim, cfg["spei_clip"]); osp = Yte_o[:, si]
            pool_obs["spei"].append(osp); pool_obs["ssi"].append(oss)
            st_obs[cid] = pd.Series(P.status_series(osp, oss, "or"), index=idx)
            vrow = pd.DataFrame({"date": idx, "cell_id": cid, "obs_spei": osp, "obs_ssi": oss})
            for m, pr in preds.items():
                fss = P.apply_ssi(pr[gi], tdoy, clim, cfg["spei_clip"]); fsp = pr[si]
                pool_fc[m]["spei"].append(fsp); pool_fc[m]["ssi"].append(fss)
                st_fc[m][cid] = pd.Series(P.status_series(fsp, fss, "or"), index=idx)
                vrow[f"{m}_spei"] = fsp; vrow[f"{m}_ssi"] = fss
            values.append(vrow)

        pd.concat(values).to_csv(os.path.join(sdir, f"values__h{h}.csv.gz"), index=False)

        OS, OG = np.concatenate(pool_obs["spei"]), np.concatenate(pool_obs["ssi"])
        for m in models_all:
            FS, FG = np.concatenate(pool_fc[m]["spei"]), np.concatenate(pool_fc[m]["ssi"])
            for logic in ("or", "and"):
                for level in LEVELS:
                    dss_pool.append(dict(horizon=h, model=m, logic=logic, level=P.LEVEL_NAMES[level],
                                         **P.validate_forecast_dss(OS, OG, FS, FG, logic, level)))

        so = pd.DataFrame(st_obs)
        sf = {m: pd.DataFrame(st_fc[m]) for m in models_all}
        reg_rows += regional_rows(so, sf, h)
        print(f"  [h={h}] done: {len(models_all)} methods, {len(ids)} cells, fractions {FRACS}", flush=True)

    return pd.DataFrame(rows), pd.DataFrame(dss_pool), pd.DataFrame(reg_rows)


def install(rf_sqrt=True):
    """Activate the patch. rf_sqrt=True reproduces the published multi-cell run, in which the
    Random Forest was patched to max_features='sqrt' before the in-domain phase."""
    import functools
    from sklearn.ensemble import RandomForestRegressor as _RF
    if rf_sqrt:
        M.RandomForestRegressor = functools.partial(_RF, max_features="sqrt")
    M.run_indomain = run_indomain_regional
    print(f"regional patch active | levels D0-D3 | fractions {FRACS} | RF max_features="
          f"{'sqrt' if rf_sqrt else 'default'}")


def _cat(out_dir, prefix):
    fs = sorted(glob.glob(os.path.join(out_dir, "parts", f"{prefix}__*.csv")))
    return pd.concat([pd.read_csv(f) for f in fs], ignore_index=True) if fs else pd.DataFrame()


def merge_regional(out_dir):
    ind = _cat(out_dir, "indomain").drop_duplicates(["cell_id", "horizon", "target", "model"])
    dss = _cat(out_dir, "dssmc").drop_duplicates(["horizon", "model", "logic", "level"])
    reg = _cat(out_dir, "dssreg").drop_duplicates(["horizon", "model", "level", "threshold"])
    ind.to_csv(os.path.join(out_dir, "metrics_indomain.csv"), index=False)
    dss.to_csv(os.path.join(out_dir, "dss_multicell.csv"), index=False)
    reg.to_csv(os.path.join(out_dir, "dss_regional_thresholds.csv"), index=False)
    print(f"[merge] indomain={len(ind)} dss={len(dss)} regional={len(reg)} "
          f"horizons={sorted(reg.horizon.unique()) if len(reg) else []}")
    return ind, dss, reg


def summarize(ind, reg, target="GWETROOT", level="D1"):
    """Persistence rank and Spearman rho (continuous skill vs regional CSI) per fraction."""
    from scipy.stats import spearmanr
    cont = ind[ind.target == target].groupby(["model", "horizon"]).SS_vs_persist.mean().rename("cs").reset_index()
    out = []
    for frac, g in reg[reg.level == level].groupby("threshold"):
        w = g.merge(cont, on=["model", "horizon"]).dropna(subset=["CSI"])
        per = {h: spearmanr(x.cs, x.CSI)[0] for h, x in w.groupby("horizon") if x.CSI.std() > 0}
        best = {h: x.loc[x.CSI.idxmax(), "model"] for h, x in g.dropna(subset=["CSI"]).groupby("horizon")}
        out.append(dict(threshold=frac, base_rate=round(g.base_rate.mean(), 3),
                        persistence_best_leads=sum(v == "Persistence" for v in best.values()),
                        n_leads=len(best), rho_pooled=round(spearmanr(w.cs, w.CSI)[0], 2),
                        **{f"rho_h{h}": round(v, 2) for h, v in sorted(per.items())}))
    return pd.DataFrame(out)


def validate_against(reg, published_csv, level="D1"):
    """Compare the 30% rows with the published dss_regional.csv."""
    if not os.path.exists(published_csv):
        print("published dss_regional.csv not found; validation skipped"); return None
    pub = pd.read_csv(published_csv)
    a = reg[(reg.threshold == 0.3)][["horizon", "model", "level", "CSI"]]
    b = pub[["horizon", "model", "level", "CSI"]]
    m = a.merge(b, on=["horizon", "model", "level"], suffixes=("_new", "_pub"))
    m["abs_diff"] = (m.CSI_new - m.CSI_pub).abs()
    grp = lambda ms: m[m.model.isin(ms)].abs_diff.max()
    print(f"30% vs published, max |dCSI|: baselines {grp(['Persistence','Climatology']):.4f} (must be 0) | "
          f"RandomForest {grp(['RandomForest']):.4f} (must be 0: CPU, fixed seeds) | "
          f"TCN/LSTM/Transformer {grp(['TCN','LSTM','Transformer']):.4f} (small differences possible: GPU)")
    return m


def score_from_values(out_dir, h, frac, level=2, logic="or"):
    """Score any aggregation fraction, level (1-4 = D0-D3) or rule offline from values__h{h}.csv.gz."""
    v = pd.read_csv(os.path.join(out_dir, "status", f"values__h{h}.csv.gz"), parse_dates=["date"])
    so = v.assign(s=P.status_series(v.obs_spei, v.obs_ssi, logic)).pivot(index="date", columns="cell_id", values="s")
    models = [c[:-5] for c in v.columns if c.endswith("_spei") and c != "obs_spei"]
    sf = {m: v.assign(s=P.status_series(v[f"{m}_spei"], v[f"{m}_ssi"], logic))
                .pivot(index="date", columns="cell_id", values="s") for m in models}
    return pd.DataFrame(regional_rows(so, sf, h, fracs=(frac,), levels=(level,)))


In [ ]:
# 4 | Activate the patch with the published settings
import importlib, tcn_drought_multicell as M, regional_patch as R
importlib.reload(M); importlib.reload(R)
R.install(rf_sqrt=True)          # reproduces the RF max_features='sqrt' patch used in the published run
C = dict(M.CONFIG_MC); C.update(data_dir=DATA_DIR, out_dir=OUT_DIR, phase='indomain')
print('seeds', C['seeds'], '| epochs', C['epochs'], '| rf_trees', C['rf_trees'], '| horizons', C['horizons'])


In [ ]:
# 5 | Run one horizon at a time (resumable). Re-run this cell after any interruption.
import time
HORIZONS = [14, 1, 7, 30]        # 14 first: its duration tells you how long the rest will take
for h in HORIZONS:
    if os.path.exists(os.path.join(OUT_DIR, 'parts', f'dssreg__h{h}.csv')):
        print(f'h={h}: already done, skipped'); continue
    t = time.time(); c = dict(C); c['only_horizons'] = [h]
    M.main(c)
    print(f'>>> h={h} finished in {(time.time()-t)/60:.1f} min\n', flush=True)
print('parts:', sorted(os.path.basename(p) for p in glob.glob(os.path.join(OUT_DIR, 'parts', '*.csv'))))


In [ ]:
# 6 | Merge, validate against the published 30% results, and summarize
ind, dss, reg = R.merge_regional(OUT_DIR)
if PUBLISHED: R.validate_against(reg, PUBLISHED)
S = R.summarize(ind, reg, target='GWETROOT', level='D1')
print(S.to_string(index=False))
pd.set_option('display.width', 200)
print(reg[reg.level == 'D1'].pivot_table(index=['threshold', 'model'], columns='horizon', values='CSI').round(3))


In [ ]:
# 7 | Package everything for download (Output panel, or right-click the file)
shutil.make_archive('/kaggle/working/results_mc_regional', 'zip', OUT_DIR)
print('ready:', '/kaggle/working/results_mc_regional.zip',
      round(os.path.getsize('/kaggle/working/results_mc_regional.zip')/1e6, 2), 'MB')


### What to send back
Upload `results_mc_regional.zip`. It contains `dss_regional_thresholds.csv` (all fractions, D0–D3),
`metrics_indomain.csv`, `dss_multicell.csv` and `status/values__h*.csv.gz` (daily values for offline scoring).

**Validation to read in cell 6:** baselines and RandomForest must show max |ΔCSI| = 0.0000 at 30%.
TCN, LSTM and Transformer may differ slightly because GPU training is not bit-for-bit deterministic.